# Почему INT4 на T4 оказался медленнее?

В одном из первых запусков на T4 проекция K/V для 2048 токенов заняла **0,108 мс** в плотном PyTorch и **33,999 мс** в Triton с INT4. Чтобы разобраться в разнице, повторим этот случай и посмотрим, что произойдёт, если распаковать те же веса заранее.

Выберите в Colab GPU T4 и нажмите **Выполнить все**. Ноутбук сам создаст веса и сожмёт их исходным способом `absmax` (`two_pass`). Сравнение `absmax` и `mse` для всех 15 размеров есть в [основном ноутбуке](llama_int4_benchmark.ipynb).

In [1]:
import torch

if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("В Colab выберите «Среда выполнения → Сменить среду выполнения → GPU T4».")
device = torch.cuda.get_device_properties(torch.cuda.current_device())
x_dtype = torch.bfloat16 if device.major >= 8 else torch.float16
dtype_label = "BF16" if x_dtype == torch.bfloat16 else "FP16"
dense_column = f"{dtype_label.lower()}_ms"

import triton
import pandas as pd

runtime = {
    "gpu": device.name,
    "compute_capability": f"{device.major}.{device.minor}",
    "torch_version": str(torch.__version__),
    "cuda_version": torch.version.cuda,
    "triton_version": triton.__version__,
}
print(f"{device.name} · {dtype_label} × INT4 и {dtype_label} × {dtype_label}")
print(f"PyTorch {torch.__version__} · CUDA {torch.version.cuda} · Triton {triton.__version__}")

NVIDIA GeForce GTX 1660 Ti · FP16 × INT4 и FP16 × FP16
PyTorch 2.10.0+cu126 · CUDA 12.6 · Triton 3.8.0


In [2]:
%%writefile task3_quantizer.py
import torch
import triton
import triton.language as tl


@triton.jit
def row_max_kernel(X, RowMax, N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    row_max = tl.full((), 0.0, tl.float32)
    for start in range(tl.cdiv(N, BLOCK_N)):
        j = start * BLOCK_N + cols
        x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(tl.abs(x), 0))
    tl.store(RowMax + row, row_max)


@triton.jit
def quantize_pack_kernel(X, RowMax, Packed, Scales,
                         N: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    block = tl.program_id(1)
    max_val = tl.load(RowMax + row)

    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    if block == 0:
        tl.store(Scales + row, scale)

    j = block * BLOCK + tl.arange(0, BLOCK)
    even = 2 * j
    odd = even + 1
    x0 = tl.load(X + row * N + even, mask=even < N, other=0.0).to(tl.float32)
    x1 = tl.load(X + row * N + odd, mask=odd < N, other=0.0).to(tl.float32)

    a0 = tl.abs(x0) * 14.0
    a1 = tl.abs(x1) * 14.0
    q0_abs = tl.full((BLOCK,), 0, tl.int32)
    q1_abs = tl.full((BLOCK,), 0, tl.int32)
    for i in tl.static_range(0, 7):
        threshold = max_val * (2 * i + 1)
        q0_abs += ((max_val > 0.0) & (a0 >= threshold)).to(tl.int32)
        q1_abs += ((max_val > 0.0) & (a1 >= threshold)).to(tl.int32)

    q0 = tl.where(x0 < 0.0, -q0_abs, q0_abs)
    q1 = tl.where(x1 < 0.0, -q1_abs, q1_abs)
    byte = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + j, byte, mask=j < N // 2)


@triton.jit
def fused_quantize_pack_kernel(X, Packed, Scales,
                               N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    j = tl.arange(0, BLOCK_N)
    x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
    max_val = tl.max(tl.abs(x), 0)
    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    tl.store(Scales + row, scale)

    # Квантизация выполняется над всеми элементами строки (пороговая
    # арифметика как в baseline), затем упаковываются пары соседей.
    a = tl.abs(x) * 14.0
    q_abs = tl.full((BLOCK_N,), 0, tl.int32)
    for k in tl.static_range(0, 7):
        threshold = max_val * (2 * k + 1)
        q_abs += ((max_val > 0.0) & (a >= threshold)).to(tl.int32)
    q = tl.where(x < 0.0, -q_abs, q_abs)

    # Gather по четным и нечетным позициям; по одной паре на байт.
    pairs = tl.arange(0, BLOCK_N // 2)
    q0 = tl.gather(q, 2 * pairs, axis=0)
    q1 = tl.gather(q, 2 * pairs + 1, axis=0)
    packed = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + pairs, packed, mask=pairs < N // 2)


def quantize_fp16_to_int4_packed(W, method="two_pass"):
    """Симметричная per-row FP16/BF16 -> packed INT4 + FP32 scales.

    method: "two_pass" (baseline) или "fused"; выходной формат одинаков.
    Контракт входа: CUDA, конечные числа, 2D, N чётное.
    Проверка конечности не выполняется внутри, чтобы не добавлять GPU sync.
    """
    assert W.is_cuda and W.ndim == 2
    assert W.dtype in (torch.float16, torch.bfloat16)
    M, N = W.shape
    assert M > 0 and N > 0 and N % 2 == 0
    W = W.contiguous()
    if method not in ("two_pass", "fused"):
        raise ValueError("method must be 'two_pass' or 'fused'")
    packed = torch.empty((M, N // 2), device=W.device, dtype=torch.int8)
    scales = torch.empty((M,), device=W.device, dtype=torch.float32)
    if method == "two_pass":
        row_max = torch.empty((M,), device=W.device, dtype=torch.float32)
        row_max_kernel[(M,)](W, row_max, N, BLOCK_N=1024, num_warps=4)
        B = 256
        quantize_pack_kernel[(M, triton.cdiv(N // 2, B))](
            W, row_max, packed, scales, N, B, num_warps=4
        )
    else:
        fused_quantize_pack_kernel[(M,)](
            W, packed, scales, N, BLOCK_N=triton.next_power_of_2(N), num_warps=4
        )
    return packed, scales


def decode_signed_int4(packed):
    """Возвращает signed целые [-7,7] в форме [M,N]."""
    p = packed.to(torch.int32)
    lo = p & 15
    hi = (p >> 4) & 15
    lo = torch.where(lo >= 8, lo - 16, lo)
    hi = torch.where(hi >= 8, hi - 16, hi)
    M, half = packed.shape
    q = torch.empty((M, 2 * half), device=packed.device, dtype=torch.int32)
    q[:, 0::2], q[:, 1::2] = lo, hi
    return q

Writing task3_quantizer.py


In [3]:
%%writefile task3_x16_matmul.py
"""Inference-only FP16 or BF16 activations @ signed, per-row INT4 weights.T.

Storage contract (shared with the quantization kernel):
    packed: int8 [N, K // 2], K must be even.
    q[n, 2*j] is the low nibble; q[n, 2*j+1] is the high nibble.
    Nibbles use two's complement; the quantizer emits values in [-7, 7].
    scales: float32 [N], W[n, k] = q[n, k] * scales[n].

No full dequantized weight matrix is allocated. Because scales are per-row,
we compute (X @ q.T) * scales in FP32, then cast to the requested output type.
All INT4 integers are exactly representable in both activation types.
"""

import torch
import triton
import triton.language as tl


@triton.jit
def _bf16_int4_matmul_kernel(
    x_ptr,
    packed_ptr,
    scales_ptr,
    y_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_pn,
    stride_pk,
    stride_s,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.program_id(1) * BLOCK_N + tl.arange(0, BLOCK_N)
    k = tl.arange(0, BLOCK_K)
    packed_k = tl.arange(0, BLOCK_K // 2)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for tile in range(tl.cdiv(K, BLOCK_K)):
        x_k = tile * BLOCK_K + k
        x = tl.load(
            x_ptr + rows[:, None] * stride_xm + x_k[None, :] * stride_xk,
            mask=(rows[:, None] < M) & (x_k[None, :] < K),
            other=0.0,
        )

        # Load every byte once: a row-major [BLOCK_N, BLOCK_K // 2] tile.
        p_k = tile * (BLOCK_K // 2) + packed_k
        packed = tl.load(
            packed_ptr + cols[:, None] * stride_pn + p_k[None, :] * stride_pk,
            mask=(cols[:, None] < N) & (p_k[None, :] < K // 2),
            other=0,
        ).to(tl.int32)

        low = packed & 0xF
        high = (packed >> 4) & 0xF
        # Two's complement: 0..7 stay positive; 8..15 become -8..-1.
        low = (low ^ 8) - 8
        high = (high ^ 8) - 8
        # join adds a final axis [low, high]; reshape interleaves each pair.
        q = tl.reshape(tl.join(low, high), (BLOCK_N, BLOCK_K))
        weights_t = tl.trans(q.to(x.dtype))
        acc = tl.dot(x, weights_t, acc)

    scale = tl.load(scales_ptr + cols * stride_s, mask=cols < N, other=0.0)
    result = acc * scale[None, :]
    tl.store(
        y_ptr + rows[:, None] * N + cols[None, :],
        result,
        mask=(rows[:, None] < M) & (cols[None, :] < N),
    )


def matmul_x16_int4(
    x: torch.Tensor,
    packed: torch.Tensor,
    scales: torch.Tensor,
    *,
    out_dtype: torch.dtype | None = None,
) -> torch.Tensor:
    """Return X @ dequantize(packed, scales).T on the current CUDA stream.

    Args:
        x: FP16 or BF16 CUDA tensor [M, K].
        packed: INT8 CUDA tensor [N, K // 2], low nibble first.
        scales: FP32 CUDA tensor [N], one finite scale per weight row.
        out_dtype: x.dtype (default) or FP32, useful for numerical validation.

    Strided input views are supported; output is contiguous [M, N]. K must
    be even, matching the quantizer's format. This function is for inference
    and does not implement autograd. On NVIDIA T4, use FP16 activations.
    NVIDIA Ampere or newer supports both FP16 and BF16 dot products.
    Per-row scales are applied after FP32 accumulation.
    """
    for name, tensor in (("x", x), ("packed", packed), ("scales", scales)):
        if not isinstance(tensor, torch.Tensor):
            raise TypeError(f"{name} must be a torch.Tensor")
    if x.ndim != 2 or packed.ndim != 2 or scales.ndim != 1:
        raise ValueError("Expected x [M, K], packed [N, K // 2], scales [N]")
    if x.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("x must have dtype torch.float16 or torch.bfloat16")
    if packed.dtype != torch.int8:
        raise TypeError("packed must have dtype torch.int8")
    if scales.dtype != torch.float32:
        raise TypeError("scales must have dtype torch.float32")
    if out_dtype is None:
        out_dtype = x.dtype
    if out_dtype not in (x.dtype, torch.float32):
        raise TypeError("out_dtype must be x.dtype or torch.float32")

    m, k = x.shape
    n = packed.shape[0]
    if k % 2:
        raise ValueError("K (in_features) must be even for two INT4 weights per byte")
    if packed.shape[1] != k // 2:
        raise ValueError("packed.shape must be [N, x.shape[1] // 2]")
    if scales.shape[0] != n:
        raise ValueError("scales must contain exactly one scale per weight row")
    if not (x.is_cuda and packed.is_cuda and scales.is_cuda):
        raise ValueError("x, packed and scales must all be CUDA tensors")
    if x.device != packed.device or x.device != scales.device:
        raise ValueError("x, packed and scales must be on the same CUDA device")

    y = torch.empty((m, n), dtype=out_dtype, device=x.device)
    if m == 0 or n == 0:
        return y
    if k == 0:
        return y.zero_()

    block_m = 16 if m <= 16 else 32 if m <= 32 else 64
    block_n, block_k = 64, 64
    grid = (triton.cdiv(m, block_m), triton.cdiv(n, block_n))
    with torch.cuda.device(x.device):
        _bf16_int4_matmul_kernel[grid](
            x, packed, scales, y,
            m, n, k,
            x.stride(0), x.stride(1),
            packed.stride(0), packed.stride(1), scales.stride(0),
            BLOCK_M=block_m,
            BLOCK_N=block_n,
            BLOCK_K=block_k,
            num_warps=4,
            num_stages=3 if torch.cuda.get_device_capability(x.device)[0] >= 8 else 1,
        )
    return y


def matmul_bf16_int4(
    x: torch.Tensor,
    packed: torch.Tensor,
    scales: torch.Tensor,
    *,
    out_dtype: torch.dtype = torch.bfloat16,
) -> torch.Tensor:
    """Multiply BF16 activations by packed INT4 weights with a BF16 or FP32 output."""
    if not isinstance(x, torch.Tensor):
        raise TypeError("x must be a torch.Tensor")
    if x.dtype != torch.bfloat16:
        raise TypeError("x must have dtype torch.bfloat16")
    if out_dtype not in (torch.bfloat16, torch.float32):
        raise TypeError("out_dtype must be torch.bfloat16 or torch.float32")
    return matmul_x16_int4(x, packed, scales, out_dtype=out_dtype)

Writing task3_x16_matmul.py

In [4]:
%%writefile task3_mma75.py
"""Turing FP16 tensor-core probe for contiguous, aligned benchmark shapes."""
import torch
import triton
import triton.language as tl


@triton.jit
def _mma16x8(a0, a1, b, c0, c1, c2, c3):
    return tl.inline_asm_elementwise(
        asm="mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {$0,$1,$2,$3}, {$4,$5}, {$6}, {$7,$8,$9,$10};",
        constraints="=f,=f,=f,=f,r,r,r,f,f,f,f",
        args=[a0, a1, b, c0, c1, c2, c3],
        dtype=(tl.float32, tl.float32, tl.float32, tl.float32),
        is_pure=True, pack=1,
    )


@triton.jit
def _packed_mma75(X, P, S, Y, M: tl.constexpr, N: tl.constexpr, K: tl.constexpr):
    # Each of four warps computes a 16-by-8 output fragment using the PTX
    # lane-to-fragment mapping. Four neighboring lanes own each 8-element K slice.
    tid = tl.inline_asm_elementwise(
        "mov.u32 $0,%tid.x;", "=r,r", [tl.arange(0, 128)],
        dtype=tl.int32, is_pure=True, pack=1,
    )
    warp = tid // 32
    lane = tid % 32
    m = tl.program_id(0) * 32 + (warp // 2) * 16
    n = tl.program_id(1) * 16 + (warp % 2) * 8
    group = lane // 4
    pair = lane % 4
    xpair = X.to(tl.pointer_type(tl.uint32))
    a_ptr = xpair + (m + group) * (K // 2) + pair
    b_ptr = P + (n + group) * (K // 2) + pair
    c0 = tl.full((128,), 0, tl.float32)
    c1 = tl.full((128,), 0, tl.float32)
    c2 = tl.full((128,), 0, tl.float32)
    c3 = tl.full((128,), 0, tl.float32)
    for tile in range(K // 32):
        for step in tl.static_range(4):
            off = tile * 16 + step * 4
            a0 = tl.load(a_ptr + off)
            a1 = tl.load(a_ptr + off + 8 * (K // 2))
            p = tl.load(b_ptr + off).to(tl.uint8).to(tl.int32)
            lo = (((p & 15) ^ 8) - 8).to(tl.float16).to(tl.uint16, bitcast=True)
            hi = ((p >> 4 ^ 8) - 8).to(tl.float16).to(tl.uint16, bitcast=True)
            b = lo.to(tl.uint32) | (hi.to(tl.uint32) << 16)
            c0, c1, c2, c3 = _mma16x8(a0, a1, b, c0, c1, c2, c3)
    col = n + pair * 2
    out = Y + (m + group) * N + col
    scale0 = tl.load(S + col)
    scale1 = tl.load(S + col + 1)
    tl.store(out, c0 * scale0)
    tl.store(out + 1, c1 * scale1)
    tl.store(out + 8 * N, c2 * scale0)
    tl.store(out + 8 * N + 1, c3 * scale1)


def matmul_fp16_int4_mma75(x, packed, scales):
    """SM75 diagnostic for the multiples-of-32/16/32 shapes used in the benchmark."""
    if x.ndim != 2 or packed.ndim != 2 or scales.ndim != 1:
        raise ValueError("Expected x[M,K], packed[N,K/2] and scales[N].")
    if not x.is_cuda or packed.device != x.device or scales.device != x.device:
        raise ValueError("All inputs must be on one CUDA device.")
    if x.dtype != torch.float16 or packed.dtype != torch.int8 or scales.dtype != torch.float32:
        raise TypeError("Expected FP16 activations, INT8 packed weights and FP32 scales.")
    m, k = x.shape
    n = packed.size(0)
    if min(m, n, k) <= 0 or m % 32 or n % 16 or k % 32:
        raise ValueError("This diagnostic kernel requires M, N, K divisible by 32, 16, 32 respectively.")
    if packed.size(1) != k // 2 or scales.size(0) != n:
        raise ValueError("Incompatible packed weight or scale shape.")
    if not all(t.is_contiguous() for t in (x, packed, scales)) or x.data_ptr() % 4:
        raise ValueError("Expected contiguous tensors and a 4-byte aligned activation pointer.")
    y = torch.empty((m, n), device=x.device, dtype=x.dtype)
    with torch.cuda.device(x.device):
        _packed_mma75[(m // 32, n // 16)](x, packed, scales, y, m, n, k, num_warps=4, num_stages=1)
    return y

Writing task3_mma75.py


In [5]:
%%writefile task3_dense_control.py
"""Dense Triton control for the packed X16 x INT4 projection benchmark.

q contains the signed, unscaled weights converted exactly to x.dtype.
Both the packed kernel and this control apply the FP32 row scale after the dot.
"""

import torch
import triton
import triton.language as tl


@triton.jit
def _dense_control_kernel(
    x_ptr, q_ptr, scales_ptr, y_ptr,
    M, N, K,
    stride_xm, stride_xk, stride_qn, stride_qk, stride_s,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.program_id(1) * BLOCK_N + tl.arange(0, BLOCK_N)
    k = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for tile in range(tl.cdiv(K, BLOCK_K)):
        x_k = tile * BLOCK_K + k
        x = tl.load(
            x_ptr + rows[:, None] * stride_xm + x_k[None, :] * stride_xk,
            mask=(rows[:, None] < M) & (x_k[None, :] < K),
            other=0.0,
        )
        q = tl.load(
            q_ptr + cols[:, None] * stride_qn + x_k[None, :] * stride_qk,
            mask=(cols[:, None] < N) & (x_k[None, :] < K),
            other=0.0,
        )
        acc = tl.dot(x, tl.trans(q), acc)

    scale = tl.load(scales_ptr + cols * stride_s, mask=cols < N, other=0.0)
    tl.store(
        y_ptr + rows[:, None] * N + cols[None, :],
        acc * scale[None, :],
        mask=(rows[:, None] < M) & (cols[None, :] < N),
    )


def matmul_x16_dense_control(x, q, scales, *, out_dtype=None):
    """Compute (x @ q.T) * scales with the launch geometry of the packed kernel."""
    if x.ndim != 2 or q.ndim != 2 or scales.ndim != 1:
        raise ValueError("Expected x [M, K], q [N, K], scales [N]")
    m, k = x.shape
    n, qk = q.shape
    if qk != k or scales.shape[0] != n:
        raise ValueError("Incompatible shapes for x, q or scales")
    if not x.is_cuda or not (x.device == q.device == scales.device):
        raise ValueError("Expected tensors on the same CUDA device")
    if x.dtype not in (torch.float16, torch.bfloat16) or q.dtype != x.dtype:
        raise TypeError("x and q must have the same 16-bit floating-point dtype")
    if scales.dtype != torch.float32:
        raise TypeError("scales must have dtype torch.float32")
    if out_dtype is None:
        out_dtype = x.dtype
    if out_dtype not in (x.dtype, torch.float32):
        raise TypeError("out_dtype must be x.dtype or torch.float32")

    y = torch.empty((m, n), dtype=out_dtype, device=x.device)
    if m == 0 or n == 0:
        return y
    if k == 0:
        return y.zero_()

    block_m = 16 if m <= 16 else 32 if m <= 32 else 64
    block_n, block_k = 64, 64
    grid = (triton.cdiv(m, block_m), triton.cdiv(n, block_n))
    with torch.cuda.device(x.device):
        _dense_control_kernel[grid](
            x, q, scales, y, m, n, k,
            x.stride(0), x.stride(1), q.stride(0), q.stride(1), scales.stride(0),
            BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
            num_warps=4,
            num_stages=3 if torch.cuda.get_device_capability(x.device)[0] >= 8 else 1,
        )
    return y

Writing task3_dense_control.py


## Одна проекция K/V, несколько вариантов умножения

У всех вариантов одна матрица активаций `X`. В таблице `W` — исходные веса, `Q` — целые числа после округления. Масштаб нужен, чтобы вернуть им величину соответствующей строки весов.

| Строка в результатах | Что получает функция | Что считает |
|---|---|---|
| PyTorch dense | Исходные веса в том же типе, что и `X` | `torch.mm(X, W.T)` |
| Triton dense Q | Те же числа `Q`, заранее распакованные и приведённые к типу `X` | `(X @ Q.T) * scales` |
| Triton packed INT4 | Числа `Q`, упакованные по два в байт | Распаковывает веса и считает `(X @ Q.T) * scales` |

Варианты `dense Q` и `packed INT4` используют `tl.dot` с одинаковыми блоками 64 × 64 × 64, настройками запуска, типами накопления и выхода. Упаковка уменьшает объём чтения и добавляет распаковку внутри кернеля. Разница во времени отражает оба эффекта вместе.

На T4 появится ещё строка **Triton packed MMA**. Там те же упакованные веса, но умножение выполняет инструкция `mma.sync`. У этого кернеля также другие размеры блоков и раскладка данных по потокам.

In [6]:
from task3_quantizer import quantize_fp16_to_int4_packed, decode_signed_int4
from task3_x16_matmul import matmul_x16_int4
from task3_dense_control import matmul_x16_dense_control

M, N, K = 2048, 512, 2048
SEED = 42
rng = torch.Generator(device="cuda").manual_seed(SEED)
w = torch.randn((N, K), device="cuda", dtype=torch.float16, generator=rng)
x = torch.randn((M, K), device="cuda", dtype=x_dtype, generator=rng)
packed, scales = quantize_fp16_to_int4_packed(w)
q = decode_signed_int4(packed).to(x_dtype).contiguous()
wt = w.to(x_dtype).T

methods = {
    "PyTorch dense": lambda: torch.mm(x, wt),
    "Triton dense Q": lambda: matmul_x16_dense_control(x, q, scales),
    "Triton packed INT4": lambda: matmul_x16_int4(x, packed, scales),
}

if (device.major, device.minor) == (7, 5):
    from task3_mma75 import matmul_fp16_int4_mma75
    methods["Triton packed MMA"] = lambda: matmul_fp16_int4_mma75(x, packed, scales)

In [7]:
@torch.inference_mode()
def check_results():
    # Берём 16 строк и 16 столбцов. Для каждого элемента суммируем все K произведений в FP32.
    rows = torch.linspace(0, M-1, 16, device=x.device).round().long()
    cols = torch.linspace(0, N-1, 16, device=x.device).round().long()
    sample_x = x[rows].float()[:, None, :]
    expected_dense = (sample_x * wt.T[cols].float()).sum(dim=-1)
    expected_int4 = (sample_x * q[cols].float()).sum(dim=-1) * scales[cols]
    for name, fn in methods.items():
        y = fn()
        assert y.shape == (M, N) and y.dtype == x.dtype
        ref = expected_dense if name == "PyTorch dense" else expected_int4
        torch.testing.assert_close(y[rows[:, None], cols].float(), ref, rtol=0.01, atol=0.025)
    control = methods["Triton dense Q"]()
    for name in ("Triton packed INT4", "Triton packed MMA"):
        if name in methods:
            torch.testing.assert_close(methods[name](), control, rtol=0.01, atol=0.025)

check_results()
print(f"Проверка пройдена · M={M}, N={N}, K={K} · {dtype_label}")

Проверка пройдена · M=2048, N=512, K=2048 · FP16


## Как измеряем время

Сначала готовим веса и проверяем результаты. К началу замеров Triton уже скомпилировал кернели. В каждом раунде `triton.testing.do_bench` использует CUDA-события, отводит 25 мс на прогрев и 150 мс на измерения каждого метода. От раунда к раунду меняем порядок, чтобы каждый метод побывал первым.

В таблице `ms` — медиана по раундам, а `time_vs_torch` — это время, делённое на время `torch.mm`. Значение выше 1 означает, что метод отработал медленнее PyTorch.

In [8]:
from statistics import median
from triton.testing import do_bench

WARMUP_MS, REP_MS = 25, 150
names = list(methods)
samples = {name: [] for name in names}
with torch.inference_mode():
    for fn in methods.values():
        fn()
    torch.cuda.synchronize()
    for idx in range(len(names)):
        for name in names[idx:] + names[:idx]:
            value = do_bench(methods[name], warmup=WARMUP_MS, rep=REP_MS, return_mode="median")
            samples[name].append(value)

ms = {name: median(values) for name, values in samples.items()}
records = []
for name, values in samples.items():
    records.append({
        "method": name, "ms": ms[name], "time_vs_torch": ms[name] / ms["PyTorch dense"],
        **{f"round_{i}_ms": v for i, v in enumerate(values, start=1)},
        "M": M, "N": N, "K": K, "dtype": dtype_label, "seed": SEED,
        "warmup_ms": WARMUP_MS, "rep_ms": REP_MS, **runtime,
    })
table = pd.DataFrame(records)
table.to_csv("llama_int4_diagnostic.csv", index=False)
display(table[["method", "ms", "time_vs_torch"]].round(4))
packed_ms, control_ms = ms["Triton packed INT4"], ms["Triton dense Q"]
print(f"Время packed / время dense Q = {packed_ms / control_ms:.3f}×")
print(f"Время dense Q / время torch.mm = {control_ms / ms['PyTorch dense']:.3f}×")
if "Triton packed MMA" in ms:
    print(f"Ускорение нового INT4-кернеля относительно исходного: {packed_ms / ms['Triton packed MMA']:.3f}×")
print("Все раунды и параметры запуска сохранены в llama_int4_diagnostic.csv")

,method,ms,time_vs_torch
0,PyTorch dense,7.7938,1.0000
1,Triton dense Q,36.4784,4.6804
2,Triton packed INT4,37.3081,4.7869
3,Triton packed MMA,4.1756,0.5358


Время packed / время dense Q = 1.023×
Время dense Q / время torch.mm = 4.680×
Ускорение нового INT4-кернеля относительно исходного: 8.935×
Все раунды и параметры запуска сохранены в llama_int4_diagnostic.csv


## Как понять, помогла ли распаковка заранее

Начните с отношения `packed / dense Q`. Если оно близко к 1, оба варианта `tl.dot` заняли примерно одинаковое время. Если оно заметно больше 1, быстрее сработал кернель с заранее распакованными весами. Если меньше 1, быстрее оказался кернель с упакованными весами. Разброс по раундам можно посмотреть в `llama_int4_diagnostic.csv`.

Для сравнения, на GTX 1660 Ti с Triton 3.7.1 получили **38,134 мс** для `packed INT4`, **37,437 мс** для `dense Q` и **4,080 мс** для `packed MMA`. На этой карте предварительная распаковка мало изменила время. Намного больший выигрыш дал другой кернель с `mma.sync`.

Эти значения получены на прогретых буферах с CUDA-событиями. После 8 прогревов каждый вариант запускали 15 раз, чередуя порядок. Все 1 048 576 элементов результата прошли проверку по FP32-эталону. Для T4 используйте таблицу из своего запуска выше.